# Lab 05: LangGraph rewrite of Lab 01

Take the agent we built from scratch in Lab 01 and rebuild it in LangGraph.
Same domain, same tools, same test queries — different wiring. Then add two
things Lab 01 couldn't easily do: checkpointing and a human-in-the-loop gate.

This notebook is the runnable companion to
[`labs/05-langgraph-rewrite/README.md`](./README.md). Read the brief first.

**Estimated time:** 90–120 minutes.
**Difficulty:** 🟡 Intermediate.
**Prerequisites:** Lab 01 completed,
[`concepts/agents/agents-vs-frameworks.md`](../../concepts/agents/agents-vs-frameworks.md),
[`tools/langgraph/snapshot-v1.0.md`](../../tools/langgraph/snapshot-v1.0.md).

> 🔴 **LangGraph is fast-changing.** The APIs used in this notebook are pinned
> to LangGraph `1.x` (verified 2026-05-23). If you're running this much later
> and something doesn't behave as described, check the
> [snapshot page](../../tools/langgraph/snapshot-v1.0.md) first.

## Step 0: Setup

We use the same canned dataset as Lab 01, but the LLM access is now mediated
through LangChain's chat model wrappers (so LangGraph can route messages
through its state). Provider-agnostic as before — OpenAI by default,
swap to Anthropic by changing `PROVIDER`.

In [ ]:
import os
import pathlib

from dotenv import load_dotenv

# Walk up to find the repo root
here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / ".env.example").exists():
        load_dotenv(parent / ".env")
        break

PROVIDER = "openai"   # or "anthropic"

assert os.getenv("OPENAI_API_KEY") or os.getenv("ANTHROPIC_API_KEY"), (
    "Set OPENAI_API_KEY or ANTHROPIC_API_KEY in .env"
)
print(f"Provider: {PROVIDER}")


**Sample output:**

```
Provider: openai
```

In [ ]:
# Initialize the LangChain chat model. This is the LangGraph-friendly entry
# point; under the hood it's the same provider client as Lab 01.

if PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
elif PROVIDER == "anthropic":
    from langchain_anthropic import ChatAnthropic
    llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature=0)
else:
    raise ValueError(f"Unknown provider: {PROVIDER}")

print(f"Model: {llm}")


**Sample output:**

```
Model: ChatOpenAI(model='gpt-4o-mini', temperature=0.0, ...)
```

## Step 1: The Lab 01 baseline (reference)

For comparison, here's the shape of Lab 01's from-scratch loop. We won't
re-run it — that's what Lab 01 is for — but it's useful to have the picture
in mind as we rebuild.

```python
def run_agent(question: str) -> dict:
    state = [{"role": "system", "content": SYSTEM_PROMPT},
             {"role": "user", "content": question}]
    for step in range(MAX_STEPS):
        msg = chat_with_tools(state, tools=schemas)
        state.append({"role": "assistant", "content": msg.content,
                      "tool_calls": [...]})
        if not msg.tool_calls:
            return {"final": msg.content, "steps": step + 1}
        for call in msg.tool_calls:
            result = execute_tool(call)
            state.append({"role": "tool", "tool_call_id": call.id,
                          "content": json.dumps(result)})
    return {"final": "[step cap]", "steps": MAX_STEPS}
```

Three things to notice, because they'll map to LangGraph parts:

1. **State** is a Python list we manually `.append(...)` to.
2. **Control flow** is `if msg.tool_calls: ... else: return`.
3. **No persistence** — when the function returns, the state vanishes.

We'll find each of these (in better form) in the LangGraph version.

## Step 2: Tools, redefined as LangChain tools

The *functions* are the same as Lab 01 — a customer lookup and a total
computer over canned data. The *decoration* is new: the `@tool` decorator
from `langchain_core.tools` turns a Python function with a docstring into a
tool the LLM can call, with the schema inferred from type hints.

In [ ]:
from langchain_core.tools import tool

# Canned data (same as Lab 01)
CUSTOMERS = {
    1001: {"id": 1001, "name": "Ada Lovelace", "plan": "pro",
           "email": "ada@example.com"},
    1002: {"id": 1002, "name": "Alan Turing", "plan": "free",
           "email": "alan@example.com"},
    1003: {"id": 1003, "name": "Grace Hopper", "plan": "pro",
           "email": "grace@example.com"},
}
ORDERS = {
    1001: [{"id": 9001, "total": 42.50}, {"id": 9002, "total": 19.99}],
    1002: [{"id": 9003, "total": 5.00}],
    1003: [],
}


@tool
def lookup_customer(email: str) -> dict:
    """Look up a customer by exact email address.

    Returns a customer record with id, name, plan, and orders. Returns an
    error dict if no customer exists with that email.
    """
    for cust in CUSTOMERS.values():
        if cust["email"].lower() == email.lower():
            return {
                **cust,
                "orders": ORDERS.get(cust["id"], []),
            }
    return {"error": "not_found", "email": email}


@tool
def compute_total(numbers: list[float]) -> dict:
    """Sum a list of numbers. Returns the total."""
    return {"total": sum(numbers), "count": len(numbers)}


tools = [lookup_customer, compute_total]
print(f"Tools defined: {[t.name for t in tools]}")
print("Schema for lookup_customer:")
print(lookup_customer.args_schema.model_json_schema())


**Sample output:**

```
Tools defined: ['lookup_customer', 'compute_total']
Schema for lookup_customer:
{'description': 'Look up a customer by exact email address.\n...',
 'properties': {'email': {'description': '...', 'title': 'Email',
                          'type': 'string'}},
 'required': ['email'],
 'title': 'lookup_customer',
 'type': 'object'}
```

The schema came from the type hints. No JSON Schema by hand, no Pydantic
boilerplate — but the same JSON Schema we'd have written by hand. This is
genuine ergonomics, not magic.

## Step 3: Build the graph

Now the core of the lab. We define a graph with two nodes — *call the model*
and *run any tools the model asked for* — and a conditional edge that decides
whether to keep looping or stop.

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langgraph.prebuilt import ToolNode, tools_condition

# Bind the tools to the model so the model knows about them. Equivalent of
# passing `tools=schemas` to `client.chat.completions.create(...)`.
llm_with_tools = llm.bind_tools(tools)


def call_model(state: MessagesState) -> dict:
    """The 'model' node: take the message history, call the LLM, return
    the new message as a state update."""
    response = llm_with_tools.invoke(state["messages"])
    # Returning a dict with the 'messages' key invokes the add_messages
    # reducer, which appends to state["messages"]. We don't mutate state.
    return {"messages": [response]}


# Build the graph
builder = StateGraph(MessagesState)
builder.add_node("model", call_model)
builder.add_node("tools", ToolNode(tools))

# Edges
builder.add_edge(START, "model")
# tools_condition is a prebuilt: returns "tools" if the last AIMessage has
# tool_calls, otherwise routes to END.
builder.add_conditional_edges(
    "model",
    tools_condition,
    {"tools": "tools", END: END},
)
builder.add_edge("tools", "model")

agent = builder.compile()
print("Graph compiled.")
print(agent.get_graph().draw_ascii())


**Sample output:**

```
Graph compiled.
       +-----------+
       | __start__ |
       +-----------+
              *
              *
              *
        +-------+
        | model |
        +-------+
        .         .
       .           .
      .             .
+-------+          +---------+
| tools |          | __end__ |
+-------+          +---------+
       *
       *
       *
   +-------+
   | model |
   +-------+
```

Three things just happened:

1. **State management** is now declarative. `MessagesState` is a TypedDict
   with one field, `messages`, that uses the `add_messages` reducer. When a
   node returns `{"messages": [m]}`, the reducer appends `m` to the existing
   list. No `.append()` calls in your code; no risk of mutating someone
   else's reference.
2. **Control flow** is a graph. The `tools_condition` function does what our
   `if msg.tool_calls:` check did in Lab 01 — but it's named, prebuilt, and
   wired into the topology declaratively.
3. **The agent is a compiled artifact.** `agent` is a runnable, not a
   function. We can invoke it, stream it, or attach a checkpointer to it.

## Step 4: Run it on the Lab 01 test queries

Same three queries as Lab 01, so the comparison is direct.

In [ ]:
from langchain_core.messages import HumanMessage

TEST_QUERIES = [
    "What's Ada Lovelace's plan? Her email is ada@example.com.",
    "How many orders does Ada have, and what's her total spend?",
    "Is alan@example.com on the pro plan?",
]

for q in TEST_QUERIES:
    print("=" * 70)
    print(f"QUERY: {q}")
    print("-" * 70)
    result = agent.invoke({"messages": [HumanMessage(content=q)]})
    # The final answer is the last AI message
    final = result["messages"][-1]
    print(f"FINAL ({len(result['messages'])} messages in trace):")
    print(f"  {final.content}")
    # Show the trace shape
    for i, m in enumerate(result["messages"]):
        role = m.__class__.__name__.replace("Message", "").lower()
        snippet = (m.content or "")[:60].replace("\n", " ")
        tool_info = ""
        if hasattr(m, "tool_calls") and m.tool_calls:
            tool_info = f" [calls: {[tc['name'] for tc in m.tool_calls]}]"
        print(f"  {i}. {role}: {snippet}{tool_info}")
    print()


**Sample output:**

```
======================================================================
QUERY: What's Ada Lovelace's plan? Her email is ada@example.com.
----------------------------------------------------------------------
FINAL (4 messages in trace):
  Ada Lovelace is on the pro plan.
  0. human: What's Ada Lovelace's plan? Her email is ada@example.com.
  1. ai:  [calls: ['lookup_customer']]
  2. tool: {"id": 1001, "name": "Ada Lovelace", "plan": "pro", ...}
  3. ai: Ada Lovelace is on the pro plan.

======================================================================
QUERY: How many orders does Ada have, and what's her total spend?
----------------------------------------------------------------------
FINAL (6 messages in trace):
  Ada has 2 orders with a total spend of $62.49.
  0. human: How many orders does Ada have, and what's her total spend?
  1. ai:  [calls: ['lookup_customer']]
  2. tool: {"id": 1001, "name": "Ada Lovelace", ...}
  3. ai:  [calls: ['compute_total']]
  4. tool: {"total": 62.49, "count": 2}
  5. ai: Ada has 2 orders with a total spend of $62.49.
```

Same behavior as Lab 01 — model picks tool, tool runs, model picks again, model
gives final answer. The trace is just a list of messages, exactly like Lab 01's
state list.

So what did we get? On the happy path: nothing. The agent answers the same
queries with the same accuracy. **The wins come when we ask for things Lab 01
couldn't do.**

## Step 5: Add a checkpointer

Lab 01's state vanishes when the function returns. If the process dies
mid-run, you start over. With a checkpointer, every state transition is
persisted, and we can resume from any point — even after the process restarts.

Real production would use `PostgresSaver` or `SqliteSaver`. For
demonstration, `InMemorySaver` works the same way without the database setup.

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# Build the SAME graph, but compile with a checkpointer this time.
checkpointer = InMemorySaver()

builder_v2 = StateGraph(MessagesState)
builder_v2.add_node("model", call_model)
builder_v2.add_node("tools", ToolNode(tools))
builder_v2.add_edge(START, "model")
builder_v2.add_conditional_edges(
    "model", tools_condition, {"tools": "tools", END: END}
)
builder_v2.add_edge("tools", "model")
agent_v2 = builder_v2.compile(checkpointer=checkpointer)

# Run with a thread_id. The same thread_id picks up the same state next time.
thread = {"configurable": {"thread_id": "demo-thread-1"}}

result_1 = agent_v2.invoke(
    {"messages": [HumanMessage(content="Look up ada@example.com")]},
    config=thread,
)
print("After first turn:")
print(f"  Last message: {result_1['messages'][-1].content[:80]}")
print(f"  Total messages in state: {len(result_1['messages'])}")


**Sample output:**

```
After first turn:
  Last message: Ada Lovelace is on the pro plan with 2 orders totaling $62.49.
  Total messages in state: 4
```

In [ ]:
# Now make a SECOND call with the same thread_id — picks up where we left off.
# In a real app this might be a new process, a new request, hours later.
# The agent remembers the prior turn because of the checkpointer.

result_2 = agent_v2.invoke(
    {"messages": [HumanMessage(content="What about alan@example.com?")]},
    config=thread,
)
print("After second turn (same thread):")
print(f"  Last message: {result_2['messages'][-1].content[:80]}")
print(f"  Total messages in state: {len(result_2['messages'])}")
print()
print("Notice the message count grew — the prior conversation is still in state.")
print("That's the checkpointer doing its job.")


**Sample output:**

```
After second turn (same thread):
  Last message: Alan Turing is on the free plan with 1 order totaling $5.00.
  Total messages in state: 8

Notice the message count grew — the prior conversation is still in state.
That's the checkpointer doing its job.
```

In [ ]:
# Verify by inspecting the checkpoint directly. We can list every checkpoint
# the saver has stored for this thread.

checkpoints = list(checkpointer.list(thread))
print(f"Number of checkpoints stored for this thread: {len(checkpoints)}")
print()
print("Each checkpoint is a snapshot of state after a node executed.")
print("If the process died at any of them, we could resume from there.")


**Sample output:**

```
Number of checkpoints stored for this thread: 8

Each checkpoint is a snapshot of state after a node executed.
If the process died at any of them, we could resume from there.
```

The number of checkpoints equals the number of state transitions, not the
number of user turns. Each node boundary is a save point.

This is the first capability that's genuinely hard to build from scratch.
Persisting state at every step, race-free across concurrent threads, with
clean resume semantics — that's a couple of weeks of work to do correctly,
and you've now got it with one parameter.

## Step 6: The `create_agent` shortcut

For the common case — a tool-using ReAct agent with no custom routing —
LangChain 1.0 added `create_agent` to skip the graph definition. It runs on
the same LangGraph runtime; it's a higher-level helper.

This is what you'd reach for in most production code that doesn't need
custom topology. The raw `StateGraph` is for when you need control.

In [ ]:
# Same agent in one line of construction
from langchain.agents import create_agent

simple_agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=(
        "You are a helpful assistant. Use the tools when appropriate, "
        "and give a concise answer when you have enough information."
    ),
)

# Same invocation pattern
simple_result = simple_agent.invoke(
    {"messages": [HumanMessage(content="What's Grace Hopper's plan? "
                                        "Her email is grace@example.com.")]}
)
print(f"Final: {simple_result['messages'][-1].content}")


**Sample output:**

```
Final: Grace Hopper is on the pro plan.
```

`create_agent` *is* a `StateGraph` under the hood — you could `.get_graph()`
on it and see the same nodes-and-edges picture as our explicit version. It
just handles the wiring for you.

**Use `create_agent` when:**

- You're building a standard tool-using agent.
- You don't need a custom state schema beyond `MessagesState`.
- You want middleware support (a feature `create_agent` exposes that raw
  `StateGraph` doesn't).

**Use raw `StateGraph` when:**

- Your control flow has branches that aren't just "tool or done."
- You're carrying state beyond messages (intermediate plans, retrieved
  context, scratchpads).
- You want to add custom nodes — guardrails, routers, summarizers — that
  the framework doesn't have prebuilt versions of.

> 📌 Migration note: `langgraph.prebuilt.create_react_agent` is the older
> equivalent of `create_agent`. It still works in LangGraph 1.x but is
> deprecated. New code should use `from langchain.agents import create_agent`.
> See the [tool snapshot](../../tools/langgraph/snapshot-v1.0.md) for
> details.

## Step 7: Human-in-the-loop with `interrupt()`

Imagine `lookup_customer` is replaced with `delete_customer` — a destructive
operation we want a human to approve before it fires. In Lab 01 we'd handle
this by returning a structured "confirmation required" error from the tool
and asking the model to re-prompt the user (Lab 02 did this for
`update_order`).

LangGraph offers a cleaner pattern: pause the agent mid-execution, surface
the proposed action to the caller, and resume once approval comes back.
This is what `interrupt()` is for.

In [ ]:
from langgraph.types import interrupt, Command


@tool
def delete_customer(email: str) -> dict:
    """Delete a customer by email. DESTRUCTIVE — requires human approval."""
    # interrupt() pauses execution and surfaces this payload to the caller.
    # When the caller resumes with Command(resume=value), `value` becomes
    # the return value of interrupt() and execution continues.
    approval = interrupt({
        "action": "delete_customer",
        "email": email,
        "message": f"About to delete customer {email}. Approve?",
    })

    # Below this line runs ONLY after a Command(resume=...) call
    if not approval:
        return {"status": "cancelled_by_user", "email": email}

    # In a real app this would mutate state. For the demo we just acknowledge.
    return {"status": "deleted", "email": email}


# Build the graph with the destructive tool included
gated_tools = [lookup_customer, delete_customer]
gated_llm = llm.bind_tools(gated_tools)


def gated_call_model(state: MessagesState) -> dict:
    return {"messages": [gated_llm.invoke(state["messages"])]}


builder_v3 = StateGraph(MessagesState)
builder_v3.add_node("model", gated_call_model)
builder_v3.add_node("tools", ToolNode(gated_tools))
builder_v3.add_edge(START, "model")
builder_v3.add_conditional_edges(
    "model", tools_condition, {"tools": "tools", END: END}
)
builder_v3.add_edge("tools", "model")
# A checkpointer is REQUIRED for interrupt() — without it, there's nowhere
# to persist state during the pause.
gated_agent = builder_v3.compile(checkpointer=InMemorySaver())
print("Gated agent compiled.")


In [ ]:
# Now drive an interaction that will trigger interrupt()
gated_thread = {"configurable": {"thread_id": "gated-thread-1"}}

# This query will lead the model to call delete_customer
initial = gated_agent.invoke(
    {"messages": [HumanMessage(content=(
        "Delete the customer whose email is alan@example.com."
    ))]},
    config=gated_thread,
)
print("After initial invocation:")
# When interrupted, the state will have an "__interrupt__" key with the payload
if "__interrupt__" in initial:
    print(f"  Interrupted! Payload: {initial['__interrupt__']}")
else:
    # If the model decided not to call delete_customer, no interrupt happens
    print(f"  No interrupt fired. Last message: {initial['messages'][-1].content[:120]}")


**Sample output:**

```
After initial invocation:
  Interrupted! Payload: (Interrupt(value={'action': 'delete_customer',
                                          'email': 'alan@example.com',
                                          'message': 'About to delete ...'}),)
```

Notice: the function `gated_agent.invoke(...)` returned. The graph isn't
deadlocked or hung — it's just paused, with everything serialized to the
checkpointer. We could exit the Python process now, restart it tomorrow,
build a fresh graph instance from the same `builder_v3`, and `.invoke()`
with `Command(resume=True)` on the same `thread_id` — and the graph would
pick up exactly where it left off, with the full message history intact.

Let's just continue in-process for the demo.

In [ ]:
# Resume with rejection
print("Resuming with REJECTION (Command(resume=False)):")
rejected = gated_agent.invoke(Command(resume=False), config=gated_thread)
print(f"  Final message: {rejected['messages'][-1].content[:150]}")


**Sample output:**

```
Resuming with REJECTION (Command(resume=False)):
  Final message: The deletion of alan@example.com was cancelled by your request.
                 Is there anything else you'd like to do?
```

In [ ]:
# Now try a new thread that approves the deletion
gated_thread_2 = {"configurable": {"thread_id": "gated-thread-2"}}

initial_2 = gated_agent.invoke(
    {"messages": [HumanMessage(content=(
        "Delete the customer whose email is alan@example.com."
    ))]},
    config=gated_thread_2,
)
print("Resuming with APPROVAL (Command(resume=True)):")
approved = gated_agent.invoke(Command(resume=True), config=gated_thread_2)
print(f"  Final message: {approved['messages'][-1].content[:150]}")


**Sample output:**

```
Resuming with APPROVAL (Command(resume=True)):
  Final message: I've deleted the customer with email alan@example.com.
                 Let me know if you need anything else.
```

This is the second capability that's genuinely hard from scratch. We have:

- A pause point inside a tool, with structured information about what's
  being proposed.
- A resume path that can happen any time later, in any process, with the
  human's decision flowing back into the agent's reasoning.
- Persistence across the pause — no in-memory state to lose.

In the Lab 01 model, the equivalent would require: a special tool return
shape, runtime code that detects it, an out-of-band channel for the
approval response, a way to persist the conversation state across the
pause, and code to splice the response back into the agent's next turn.
It's possible. It's a lot of code, and it's easy to get wrong.

### Design note: tool-level vs. node-level interrupts

We put the `interrupt()` *inside* the `delete_customer` tool, which keeps
the graph topology simple — there's no separate approval node. The
alternative is a dedicated `human_approval` node placed between `model` and
`tools` that always runs and decides whether to interrupt:

```python
def human_approval(state):
    last = state["messages"][-1]
    if any(tc["name"] in DESTRUCTIVE_TOOLS for tc in last.tool_calls):
        approval = interrupt({"tool_calls": last.tool_calls})
        if not approval:
            return Command(goto=END)
    return state
```

Both patterns work. The **tool-level** form keeps the responsibility next to
the tool that needs it. The **node-level** form centralizes the policy and
makes it easier to add cross-cutting rules (e.g., "always interrupt for
*any* tool call from this user"). Pick based on which constraint is more
likely to change.

## Step 8 (stretch): Streaming intermediate state

`.invoke()` waits for the whole run to finish. `.stream()` yields each node's
output as it happens — useful for UIs where you want to display the agent's
thinking live.

In [ ]:
print("Streaming a run, showing each node's output:")
print("-" * 70)
stream_thread = {"configurable": {"thread_id": "stream-thread"}}
for event in agent_v2.stream(
    {"messages": [HumanMessage(content=(
        "Look up grace@example.com and tell me her plan and order count."
    ))]},
    config=stream_thread,
):
    for node_name, node_output in event.items():
        last_msg = node_output["messages"][-1]
        kind = last_msg.__class__.__name__
        snippet = (getattr(last_msg, "content", "") or "")[:80]
        tool_info = ""
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            tool_info = f" → {[tc['name'] for tc in last_msg.tool_calls]}"
        print(f"  [{node_name}] {kind}: {snippet!r}{tool_info}")


**Sample output:**

```
Streaming a run, showing each node's output:
----------------------------------------------------------------------
  [model] AIMessage: '' → ['lookup_customer']
  [tools] ToolMessage: '{"id": 1003, "name": "Grace Hopper", "plan": "pro",...'
  [model] AIMessage: 'Grace Hopper is on the pro plan with 0 orders.'
```

You can see the agent's three-step trace as it happens, node by node.
The same information was in `result["messages"]` after `.invoke()`; streaming
just emits it incrementally.

This is useful for showing the user *what the agent is doing* in real time —
a UX feature that's awkward to build on top of a from-scratch loop.

## ✓ Lab complete

You've now rebuilt Lab 01's agent in LangGraph, observed the parts that
correspond one-to-one with the from-scratch version, and extended it with
two capabilities that are hard to add from scratch:

| Lab 01 (from scratch) | Lab 05 (LangGraph) |
|---|---|
| `state = []; state.append(...)` | `MessagesState` + `add_messages` reducer |
| `for step in range(MAX_STEPS):` | `StateGraph` with `model ↔ tools` edges |
| `if msg.tool_calls:` | `tools_condition` conditional edge |
| `execute_tool(call)` | `ToolNode(tools)` |
| Function returns, state vanishes | `compile(checkpointer=InMemorySaver())` |
| Special return shape for approval | `interrupt(...)` + `Command(resume=...)` |
| (Manual logging) | `.stream(...)` for live trace |

**The header takeaway:** none of these are improvements to the *agent*. The
agent is the same model, picking the same actions, on the same tools. What
changed is the *runtime around the agent*. That's exactly the right way to
think about frameworks — they're runtimes, not intelligence.

### What to do next

- 🧠 **Take the quiz:** [`quizzes/foundations/langgraph-basics.md`](../../quizzes/foundations/langgraph-basics.md)
- 📖 **Reread:** [`concepts/agents/agents-vs-frameworks.md`](../../concepts/agents/agents-vs-frameworks.md)
  with the lab fresh in mind. The decision table should make more sense now.
- 🗺 **Move on** to Path 02 (Agentic RAG) or Path 03 (Multi-Agent Systems).
  The patterns you just learned generalize directly.

### Going deeper (optional)

- Read the [LangGraph migration guide](https://docs.langchain.com/oss/python/migrate/langgraph-v1)
  to see the full deprecation table.
- Look up [`langgraph.checkpoint.postgres.PostgresSaver`](https://docs.langchain.com/oss/python/langgraph/add-memory) —
  the production-grade checkpointer. The API is identical to `InMemorySaver`;
  swap one line and you're persisting to a database.
- Skim [Building effective agents](https://www.anthropic.com/engineering/building-effective-agents) (Anthropic, 2024)
  for a thoughtful argument that *simpler* patterns (chains, routers) often
  beat full agentic flow — a useful counterweight to "use the framework
  everywhere."

### A confession about scope

This lab deliberately *doesn't* cover multi-agent topologies, LangSmith
tracing, custom state schemas, or middleware. Those are real LangGraph
features; they're not Foundations material. We'll meet them when their
corresponding paths land (Multi-Agent → Path 03, Observability → Path 06,
Custom state → covered as needed in later labs).